<a href="https://colab.research.google.com/github/tomlongcool/plm_tools/blob/main/HAS_protein.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
# 1. 加载模型
# =========================
# 如果你有自己微调好的 TemBERTure / ProtBERT 分类模型，把这里换成本地路径即可
# 例如 MODEL_NAME = "./my_finetuned_protbert_model"
MODEL_NAME = "Rostlab/prot_bert_bfd"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    do_lower_case=False
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    output_attentions=True
)

model.eval()


# =========================
# 2. 蛋白序列预处理
# =========================
def format_protein_sequence(seq):
    """
    ProtBERT 一般要求氨基酸之间用空格分隔。
    例如: MKTFFV -> M K T F F V
    """
    seq = seq.replace(" ", "")
    seq = seq.replace("U", "X").replace("Z", "X").replace("O", "X").replace("B", "X")
    return " ".join(list(seq))


# 示例蛋白序列
sequence = "MKTFFVAGVILALALAGALAAPAA"

formatted_seq = format_protein_sequence(sequence)

inputs = tokenizer(
    formatted_seq,
    return_tensors="pt",
    add_special_tokens=True
)


# =========================
# 3. 前向传播，获取 attentions
# =========================
with torch.no_grad():
    outputs = model(
        **inputs,
        output_attentions=True
    )

attentions = outputs.attentions

print("Number of layers:", len(attentions))
print("Attention shape of one layer:", attentions[0].shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: Rostlab/prot_bert_bfd
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initial

Number of layers: 30
Attention shape of one layer: torch.Size([1, 16, 26, 26])


In [2]:
# =========================
# 4. 多层、多头 attention 聚合
# =========================

# attentions 是一个 tuple，每个元素对应一层
# 每层 shape: [batch, heads, tokens, tokens]
# 堆叠后 shape: [layers, batch, heads, tokens, tokens]
attention_stack = torch.stack(attentions)

# 取第 0 个样本
# 取 CLS token 对所有 token 的 attention
# 对所有 layer 和 head 求平均
#
# attention_stack shape:
# [layers, batch, heads, query_token, key_token]
#
# CLS token 通常在第 0 个位置
cls_attention = attention_stack[:, 0, :, 0, :]

# 对 layers 和 heads 求平均
# 得到每个 token 的 attention score
token_attention_scores = cls_attention.mean(dim=(0, 1))

# 转成 numpy
token_attention_scores = token_attention_scores.cpu().numpy()


# =========================
# 5. 去除特殊 token，只保留氨基酸 token
# =========================
input_ids = inputs["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

result = []

for i, token in enumerate(tokens):
    # 跳过特殊 token
    if token in tokenizer.all_special_tokens:
        continue

    result.append({
        "token_index": i,
        "residue": token,
        "attention_score": float(token_attention_scores[i])
    })

df = pd.DataFrame(result)

# =========================
# 6. 定义 high attention score
# =========================
# 这里用 top 10% 作为 high attention residues
# 也可以改成 top 5%、top 20%，取决于你的定义
threshold = df["attention_score"].quantile(0.90)

df["is_high_attention"] = df["attention_score"] >= threshold

print(df)

    token_index residue  attention_score  is_high_attention
0             1       M         0.108934               True
1             2       K         0.030700               True
2             3       T         0.027521              False
3             4       F         0.019917              False
4             5       F         0.017419              False
5             6       V         0.015235              False
6             7       A         0.013492              False
7             8       G         0.012898              False
8             9       V         0.011398              False
9            10       I         0.013491              False
10           11       L         0.011443              False
11           12       A         0.010595              False
12           13       L         0.011555              False
13           14       A         0.010535              False
14           15       L         0.011546              False
15           16       A         0.008640

In [3]:
df.to_excel('result.xlsx')